The primary question is using a qwen model 

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

In [2]:
%%capture
! pip uninstall unsloth unsloth_zoo -y
! pip install git+https://github.com/unslothai/unsloth-zoo.git --no-deps
! pip install git+https://github.com/unslothai/unsloth.git --no-deps

In [3]:
import os
# Note that to get best performance on A100, we needed to install causal-conv1d
# For this to not take too long, we had to downgrade to torch 2.8 (from torch 2.9)
# This means we wouldn't be able to use torch's grouped_mm here
# so we fallback to unsloth triton kernels for MoE
# We need to disable autotuning to save both time and memory for the colab notebook.
# If you are trying this elsewhere, we might recommend
# you install Flash Attention, Flash Linear Attention and CausalConv1d with torch 2.9
# `!uv pip install --no-build-isolation flash-attn flash-linear-attention causal_conv1d==1.6.`
# You can even try playing around with the below env var for faster performance but make sure you have enough VRAM to try autotuning.
os.environ['UNSLOTH_MOE_DISABLE_AUTOTUNE']='1'

In [1]:
!pip install  -U transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 146.7 MB/s eta 0:00:0000:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [14]:
from huggingface_hub import snapshot_download
snapshot_download(
    "unsloth/Qwen3.5-4B",
    local_dir="MODELLLLL_Qwen3.5-4B", 
    token="YOUR_HF_TOKEN"

)


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

'/content/MODELLLLL_Qwen3.5-4B'

In [8]:
print("Done!")

Done!


In [16]:
import  os 
os.listdir("MODELLLLL_Qwen3.5-4B")

['.cache',
 'model.safetensors.index.json',
 'processor_config.json',
 'config.json',
 'tokenizer_config.json',
 'tokenizer.json',
 'vocab.json',
 'preprocessor_config.json',
 'README.md',
 '.gitattributes',
 'model.safetensors-00001-of-00002.safetensors',
 'model.safetensors-00002-of-00002.safetensors',
 'merges.txt',
 'video_preprocessor_config.json',
 'LICENSE',
 'chat_template.jinja']

### Unsloth

In [2]:
# from transformers import AutoProcessor, AutoModelForCausalLM

# processor = AutoProcessor.from_pretrained("Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled")
# model = AutoModelForCausalLM.from_pretrained("Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled")


In [ ]:
# from unsloth import FastLanguageModel
# import torch

# max_seq_length = 92048 # Can increase for longer reasoning traces
# lora_rank = 16 # Larger rank = smarter, but slower

# model, processor = FastLanguageModel.from_pretrained(
#     "Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled", # This is a very big model, might take a while for downloading
#     max_seq_length = max_seq_length,
#     load_in_4bit = False,
#     #fast_inference = False, # Not supported for MoE (yet!) ,
#     hf_token = "YOUR_HF_TOKEN"

# )
# tokenizer = processor.tokenizer # To tokenize text

In [5]:

# model, processor = FastLanguageModel.from_pretrained(
#     "Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled", # This is a very big model, might take a while for downloading
#     max_seq_length = max_seq_length,
#     load_in_4bit = False,
#     #fast_inference = False, # Not supported for MoE (yet!) ,
#     #hf_token = "YOUR_HF_TOKEN"
# )


In [6]:
# model.save_pretrained("MODEL")


In [ ]:
# processor.save_pretrained("MODEL")

('MODEL/tokenizer_config.json',
 'MODEL/chat_template.jinja',
 'MODEL/tokenizer.json')

In [6]:
import kagglehub
kagglehub.login()

In [17]:
handle = 'barnobarno/Qwen3.5-4B/Transformers/Unsloth'
local_model_dir = "MODELLLLL_Qwen3.5-4B"

kagglehub.model_upload(handle, local_model_dir)


Uploading Model https://www.kaggle.com/models/barnobarno/Qwen3.5-4B/Transformers/Unsloth ...
Model 'Qwen3.5-4B' does not exist or access is forbidden for user 'barnobarno'. Creating or handling Model...
Model 'Qwen3.5-4B' Created.
Starting upload for file MODELLLLL_Qwen3.5-4B/model.safetensors.index.json





Uploading: 100%|██████████| 76.2k/76.2k [00:00<00:00, 83.8kB/s]

Upload successful: MODELLLLL_Qwen3.5-4B/model.safetensors.index.json (74KB)
Starting upload for file MODELLLLL_Qwen3.5-4B/processor_config.json





Uploading: 100%|██████████| 1.30k/1.30k [00:00<00:00, 1.65kB/s]

Upload successful: MODELLLLL_Qwen3.5-4B/processor_config.json (1KB)
Starting upload for file MODELLLLL_Qwen3.5-4B/config.json


...




Uploading: 100%|██████████| 11.5k/11.5k [00:00<00:00, 15.3kB/s]

Upload successful: MODELLLLL_Qwen3.5-4B/LICENSE (11KB)
Starting upload for file MODELLLLL_Qwen3.5-4B/chat_template.jinja





Uploading: 100%|██████████| 7.82k/7.82k [00:00<00:00, 9.71kB/s]

Upload successful: MODELLLLL_Qwen3.5-4B/chat_template.jinja (8KB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/barnobarno/Qwen3.5-4B/Transformers/Unsloth


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "gate_up_proj", #Enable LoRA on MoE layers
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = True, # Reduces memory usage
    random_state = 3407,
    bias = "none",
)

Unsloth: Detected MoE model with num_experts = 256 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'gate_up_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:212: UserWarning: Unsupported layer type '<class 'transformers.models.qwen3_5_moe.modeling_qwen3_5_moe.Qwen3_5MoeExperts'>' encountered, proceed at your own risk.
  warnings.warn(f"Unsupported layer type '{type(module)}' encountered, proceed at your own risk.", UserWarning)


Unsloth: Making `model.base_model.model.model.language_model` require gradients


<a name="Data"></a>
### Data Prep
We now use the `Qwen 3.5` format for conversation style finetunes. We use the [Open Math Reasoning](https://huggingface.co/datasets/unsloth/OpenMathReasoning-mini) dataset which was used to win the [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard) (AI Mathematical Olympiad - Progress Prize 2) challenge! We sample 10% of verifiable reasoning traces that used DeepSeek R1, and which got > 95% accuracy.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")

We now convert the reasoning dataset into conversational format:

In [ ]:
def generate_conversation(examples):
    problems  = examples["problem"]
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }

dataset = dataset.map(generate_conversation, batched = True)

We now have to apply the chat template for `Qwen 3.5` onto the conversations, and save it to `text`.

In [ ]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Let's see how the chat template did!

In [ ]:
dataset[100]['text']

'<|im_start|>user\nOn a wall, there are two clocks with the same shape (radius) and the same speed, but they may not show the same hour. The minimum distance between the edges of their hands is \\( m \\), and the maximum distance is \\( M \\). What is the distance between their centers?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their 

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 50,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/46 [00:00<?, ? examples/s]

In [ ]:
dataset[100]['text']

'<|im_start|>user\nOn a wall, there are two clocks with the same shape (radius) and the same speed, but they may not show the same hour. The minimum distance between the edges of their hands is \\( m \\), and the maximum distance is \\( M \\). What is the distance between their centers?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their 

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 50,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n<think>",
)

Map (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/19252 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|im_start|>user\nOn a wall, there are two clocks with the same shape (radius) and the same speed, but they may not show the same hour. The minimum distance between the edges of their hands is \\( m \\), and the maximum distance is \\( M \\). What is the distance between their centers?<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their 

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                     \nOkay, so I have this problem here about two clocks on a wall. Both clocks have the same radius and the same speed, but they might show different times. The minimum distance between the edges of their hands is m, and the maximum is M. I need to find the distance between their centers. Hmm, let\'s break this down.\n\nFirst, let me visualize the situation. Both clocks are circular with the same radius, let\'s say radius r. The hands of each clock are moving at the same speed, so their minute and hour hands move at the same rates. However, they might not be showing the same time, which means their hands could be pointing in different directions. The edges of their hands—so the tips of the hands—are what we\'re concerned with here. The minimum distance between these tips is m, and the maximum is M. We need to find the distance between the centers of the two clocks.\n\nWait, but how are the clocks arranged on the wall? A

In [ ]:
# Compilation can take 2-3 minutes of time, so please be patient :)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None, 'pad_token_id': 248055}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,252 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 931,102,720 of 36,038,284,656 (2.58% trained)


Step,Training Loss
5,0.413158
10,0.406823
15,0.305829
20,0.412195
25,0.415526
30,0.389197
35,0.345411
40,0.394056
45,0.389188
50,0.379886


TrainOutput(global_step=50, training_loss=0.38512696504592897, metrics={'train_runtime': 271.3317, 'train_samples_per_second': 0.184, 'train_steps_per_second': 0.184, 'total_flos': 2.1147204848563296e+16, 'train_loss': 0.38512696504592897, 'epoch': 0.0025971327654269686})

Let's check if the model has learnt to follow the custom format:

In [ ]:
messages = [
    {"role" : "user", "content" : "Continue the sequence: 1, 1, 2, 3, 5, 8,"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1000, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Thinking Process:

1.  **Analyze the Request:** The user wants me to continue the sequence: 1, 1, 2, 3, 5, 8, ...

2.  **Identify the Pattern:**
    *   Look at the numbers: 1, 1, 2, 3, 5, 8.
    *   Check the relationship between consecutive terms.
    *   1 + 1 = 2
    *   1 + 2 = 3
    *   2 + 3 = 5
    *   3 + 5 = 8
    *   This is the Fibonacci sequence, where each number is the sum of the two preceding ones.

3.  **Calculate the Next Term:**
    *   The last two terms are 5 and 8.
    *   Next term = 5 + 8 = 13.

4.  **Formulate the Output:**
    *   State the next number in the sequence.
    *   Optionally explain the pattern (Fibonacci sequence).

5.  **Final Decision:** Provide the next number clearly.

    *   Next number: 13.
    *   Sequence so far: 1, 1, 2, 3, 5, 8, 13.

6.  **Construct Response:** "13" or "The next number is 13." Let's keep it direct but helpful. "13" is the core answer. I'll provide just the number or a short sentence. Usually, for sequence continuation,

Yes it did follow the formatting! Great! Let's remove some items before the GRPO step

In [ ]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

5349

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).